In [ ]:
# =====================================================================
# =====================================================================
#
#     EXTERNAL / COMPLETELY UNSEEN TEST
#
#     Generate NEW McStas diffraction patterns at lambda/dlambda
#     conditions which NEVER appeared in the original dataset.
#
#     Then:
#
#       NEW McStas simulation
#               ↓
#       2D diffraction image
#               ↓
#       radial integration I(2theta)
#               ↓
#       exact training preprocessing
#               ↓
#       saved PyTorch model
#               ↓
#       MATERIAL PREDICTION
#
# =====================================================================
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

from pathlib import Path
import subprocess
import shutil
import json
import random
import contextlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# =====================================================================
# 1. USER SETTINGS
# =====================================================================


# ---------------------------------------------------------------------
# Dataset root generated previously
# ---------------------------------------------------------------------

DATASET_ROOT = Path(
    "multi_material_diffraction_dataset"
)


# ---------------------------------------------------------------------
# Number of NEW lambda/dlambda conditions
#
# Each condition will be simulated for EVERY material.
#
# Example:
#
#   10 materials × 12 unseen conditions = 120 simulations
#
# ---------------------------------------------------------------------

N_UNSEEN_CONDITIONS = 12


# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

SEED = 1234


# ---------------------------------------------------------------------
# If a new external simulation already exists:
#
# False = reuse it
# True  = simulate again
# ---------------------------------------------------------------------

OVERWRITE_EXISTING = False


# ---------------------------------------------------------------------
# Save fresh 2D patterns as well
# ---------------------------------------------------------------------

SAVE_2D_PATTERNS = True


# ---------------------------------------------------------------------
# Number of example predictions to plot
# ---------------------------------------------------------------------

N_EXAMPLES_TO_PLOT = 10


# =====================================================================
# 2. RANDOM SEEDS
# =====================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# =====================================================================
# 3. FIND THE ORIGINAL DATASET
# =====================================================================

metadata_candidates = list(

    DATASET_ROOT.rglob(
        "master_metadata.csv"
    )
)


if len(metadata_candidates) == 0:

    raise FileNotFoundError(
        "\nCould not find master_metadata.csv under:\n"
        f"{DATASET_ROOT.resolve()}"
    )


# newest sweep
metadata_file = max(

    metadata_candidates,

    key=lambda p: p.stat().st_mtime
)


sweep_root = metadata_file.parent


print("=" * 80)
print("ORIGINAL DATASET")
print("=" * 80)

print()

print(
    sweep_root.resolve()
)


# =====================================================================
# 4. IMPORTANT FILES
# =====================================================================

checkpoint_file = (

    sweep_root
    /
    "ML_material_classifier"
    /
    "best_material_classifier.pt"
)


settings_file = (

    sweep_root
    /
    "settings.json"
)


materials_file = (

    sweep_root
    /
    "materials.csv"
)


instrument_file = (

    sweep_root
    /
    "instrument_snapshot.instr"
)


for required_file in [

    checkpoint_file,
    settings_file,
    materials_file,
    instrument_file

]:

    if not required_file.exists():

        raise FileNotFoundError(

            f"\nRequired file missing:\n"
            f"{required_file.resolve()}"
        )


# =====================================================================
# 5. OUTPUT DIRECTORY FOR THIS EXTERNAL TEST
# =====================================================================

external_test_dir = (

    sweep_root
    /
    "external_unseen_test"
)


runs_dir = (

    external_test_dir
    /
    "mcstas_runs"
)


patterns_2d_dir = (

    external_test_dir
    /
    "patterns_2D"
)


profiles_1d_dir = (

    external_test_dir
    /
    "profiles_1D"
)


runs_dir.mkdir(
    parents=True,
    exist_ok=True
)

patterns_2d_dir.mkdir(
    parents=True,
    exist_ok=True
)

profiles_1d_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =====================================================================
# 6. LOAD ORIGINAL METADATA + SETTINGS
# =====================================================================

original_df = pd.read_csv(
    metadata_file
)


materials_df = pd.read_csv(
    materials_file
)


with open(
    settings_file,
    "r"
) as f:

    settings = json.load(
        f
    )


# =====================================================================
# 7. DEVICE
# =====================================================================

if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif torch.backends.mps.is_available():

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )


USE_AMP = (
    device.type == "cuda"
)


print()

print(
    f"Device: {device}"
)


# =====================================================================
# 8. LOAD TRAINED CHECKPOINT
# =====================================================================

try:

    checkpoint = torch.load(

        checkpoint_file,

        map_location=device,

        weights_only=False
    )

except TypeError:

    checkpoint = torch.load(

        checkpoint_file,

        map_location=device
    )


class_names = checkpoint[
    "class_names"
]


class_to_idx = checkpoint[
    "class_to_idx"
]


theta_reference = np.asarray(

    checkpoint[
        "theta_deg"
    ],

    dtype=np.float64
)


train_mean = float(

    checkpoint[
        "train_global_mean"
    ]
)


train_std = float(

    checkpoint[
        "train_global_std"
    ]
)


PROFILE_KEY = checkpoint[
    "profile_key"
]


INTENSITY_PERCENTILE = float(

    checkpoint[
        "intensity_percentile"
    ]
)


MAX_NORMALIZED_INTENSITY = float(

    checkpoint[
        "max_normalized_intensity"
    ]
)


LOG_COMPRESSION = float(

    checkpoint[
        "log_compression"
    ]
)


num_classes = len(
    class_names
)


print()

print("=" * 80)
print("TRAINED CLASSES")
print("=" * 80)

for i, material in enumerate(
    class_names
):

    print(
        f"{i:2d}. {material}"
    )


# =====================================================================
# 9. GET NCRYSTAL FILE FOR EACH TRAINED MATERIAL
# =====================================================================

material_to_ncrystal = dict(

    zip(

        materials_df[
            "material"
        ],

        materials_df[
            "ncrystal_file"
        ]
    )
)


for material in class_names:

    if material not in material_to_ncrystal:

        raise RuntimeError(

            f"\nMaterial '{material}' exists in the trained model "
            "but was not found in materials.csv"
        )


# =====================================================================
# 10. ORIGINAL PARAMETER GRID
# =====================================================================

original_lambdas = np.sort(

    original_df[
        "lambda_A"
    ].unique()
)


original_dlambdas = np.sort(

    original_df[
        "dlambda_A"
    ].unique()
)


print()

print("=" * 80)
print("ORIGINAL PARAMETER GRID")
print("=" * 80)

print()

print(
    "lambda:"
)

print(
    original_lambdas
)

print()

print(
    "dlambda:"
)

print(
    original_dlambdas
)


# =====================================================================
# 11. CREATE TRULY UNSEEN VALUES
# =====================================================================
#
# We use MIDPOINTS between original grid values.
#
# Example:
#
# original:
#
#     1.50
#     1.60
#
# unseen:
#
#     1.55
#
#
# Therefore these values were NEVER simulated previously.
#
# =====================================================================


unseen_lambda_candidates = (

    original_lambdas[:-1]
    +
    original_lambdas[1:]

) / 2.0


unseen_dlambda_candidates = (

    original_dlambdas[:-1]
    +
    original_dlambdas[1:]

) / 2.0


print()

print("=" * 80)
print("POSSIBLE NEW / UNSEEN VALUES")
print("=" * 80)

print()

print(
    "Unseen lambda candidates:"
)

print(
    unseen_lambda_candidates
)

print()

print(
    "Unseen dlambda candidates:"
)

print(
    unseen_dlambda_candidates
)


# =====================================================================
# 12. ALL POSSIBLE UNSEEN CONDITIONS
# =====================================================================

possible_conditions = [

    (
        float(lam),
        float(dlam)
    )

    for lam in unseen_lambda_candidates

    for dlam in unseen_dlambda_candidates
]


# =====================================================================
# 13. CHECK AGAINST EVERY ORIGINAL CONDITION
# =====================================================================

original_conditions = [

    (
        float(lam),
        float(dlam)
    )

    for lam, dlam

    in zip(

        original_df[
            "lambda_A"
        ],

        original_df[
            "dlambda_A"
        ]
    )
]


def condition_exists(
    condition,
    old_conditions
):

    lam_new, dlam_new = condition

    for lam_old, dlam_old in old_conditions:

        if (

            np.isclose(
                lam_new,
                lam_old
            )

            and

            np.isclose(
                dlam_new,
                dlam_old
            )

        ):

            return True

    return False


possible_conditions = [

    condition

    for condition

    in possible_conditions

    if not condition_exists(

        condition,

        original_conditions
    )
]


if len(
    possible_conditions
) < N_UNSEEN_CONDITIONS:

    raise RuntimeError(

        "\nNot enough unseen midpoint conditions are available.\n"

        f"Available: {len(possible_conditions)}\n"

        f"Requested: {N_UNSEEN_CONDITIONS}"
    )


# =====================================================================
# 14. RANDOMLY SELECT NEW CONDITIONS
# =====================================================================

rng = np.random.default_rng(
    SEED
)


selected_indices = rng.choice(

    len(
        possible_conditions
    ),

    size=N_UNSEEN_CONDITIONS,

    replace=False
)


unseen_conditions = [

    possible_conditions[
        i
    ]

    for i in selected_indices
]


unseen_conditions = sorted(
    unseen_conditions
)


print()

print("=" * 80)
print("SELECTED COMPLETELY UNSEEN CONDITIONS")
print("=" * 80)

print()


for i, (lam, dlam) in enumerate(
    unseen_conditions,
    start=1
):

    print(

        f"{i:2d}. "

        f"lambda = "
        f"{lam:.5f} Å    "

        f"dlambda = "
        f"{dlam:.5f} Å"
    )


# =====================================================================
# 15. DOUBLE CHECK
# =====================================================================

for condition in unseen_conditions:

    assert not condition_exists(

        condition,

        original_conditions
    )


print()

print(
    "✓ None of these conditions exists in the original dataset."
)


# =====================================================================
# 16. McSTAS SOFTWARE CHECK
# =====================================================================

mcrun = shutil.which(
    "mcrun"
)


if mcrun is None:

    raise RuntimeError(

        "\nmcrun was not found.\n"
        "Make sure the PaNRAID / McStas environment is active."
    )


# =====================================================================
# 17. REBUILD THE EXACT MODEL ARCHITECTURE
# =====================================================================


class SEBlock1D(
    nn.Module
):

    def __init__(
        self,
        channels,
        reduction=8
    ):

        super().__init__()

        hidden = max(

            channels
            //
            reduction,

            8
        )

        self.pool = nn.AdaptiveAvgPool1d(
            1
        )

        self.fc = nn.Sequential(

            nn.Conv1d(
                channels,
                hidden,
                kernel_size=1
            ),

            nn.SiLU(),

            nn.Conv1d(
                hidden,
                channels,
                kernel_size=1
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        x
    ):

        return (

            x

            *

            self.fc(
                self.pool(
                    x
                )
            )
        )


class ResidualBlock1D(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        dropout=0.05
    ):

        super().__init__()

        padding = (
            kernel_size
            //
            2
        )

        self.conv1 = nn.Conv1d(

            in_channels,
            out_channels,

            kernel_size=kernel_size,

            stride=stride,

            padding=padding,

            bias=False
        )

        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        self.act = nn.SiLU()

        self.conv2 = nn.Conv1d(

            out_channels,
            out_channels,

            kernel_size=kernel_size,

            padding=padding,

            bias=False
        )

        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        self.dropout = nn.Dropout1d(
            dropout
        )

        self.se = SEBlock1D(
            out_channels
        )

        if (

            stride != 1

            or

            in_channels != out_channels

        ):

            self.shortcut = nn.Sequential(

                nn.Conv1d(

                    in_channels,
                    out_channels,

                    kernel_size=1,

                    stride=stride,

                    bias=False
                ),

                nn.BatchNorm1d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()


    def forward(
        self,
        x
    ):

        identity = self.shortcut(
            x
        )

        out = self.conv1(
            x
        )

        out = self.bn1(
            out
        )

        out = self.act(
            out
        )

        out = self.dropout(
            out
        )

        out = self.conv2(
            out
        )

        out = self.bn2(
            out
        )

        out = self.se(
            out
        )

        out = (
            out
            +
            identity
        )

        return self.act(
            out
        )


class DiffractionResNet1D(
    nn.Module
):

    def __init__(
        self,
        num_classes,
        dropout=0.30
    ):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv1d(

                1,
                32,

                kernel_size=15,

                stride=2,

                padding=7,

                bias=False
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.SiLU()
        )

        self.features = nn.Sequential(

            ResidualBlock1D(
                32,
                32,
                7,
                1
            ),

            ResidualBlock1D(
                32,
                64,
                7,
                2
            ),

            ResidualBlock1D(
                64,
                64,
                5,
                1
            ),

            ResidualBlock1D(
                64,
                128,
                5,
                2
            ),

            ResidualBlock1D(
                128,
                128,
                3,
                1
            ),

            ResidualBlock1D(
                128,
                192,
                3,
                2
            ),

            ResidualBlock1D(
                192,
                192,
                3,
                1
            )
        )

        self.avg_pool = nn.AdaptiveAvgPool1d(
            1
        )

        self.max_pool = nn.AdaptiveMaxPool1d(
            1
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                384,
                128
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        x = self.stem(
            x
        )

        x = self.features(
            x
        )

        x = torch.cat(

            [

                self.avg_pool(
                    x
                ),

                self.max_pool(
                    x
                )

            ],

            dim=1
        )

        return self.classifier(
            x
        )


model = DiffractionResNet1D(

    num_classes=num_classes,

    dropout=0.30
)


model.load_state_dict(

    checkpoint[
        "model_state_dict"
    ]
)


model = model.to(
    device
)


model.eval()


print()

print(
    "✓ Best trained model loaded."
)


# =====================================================================
# 18. READ McSTAS 2D MONITOR
# =====================================================================

def read_mcstas_2d_intensity(
    filename
):

    filename = Path(
        filename
    )

    lines = filename.read_text().splitlines()

    data_start = None
    data_end = None

    for i, line in enumerate(
        lines
    ):

        if line.startswith(
            "# Data"
        ):

            data_start = i + 1

            break


    if data_start is None:

        raise ValueError(

            f"Could not find '# Data' in:\n"
            f"{filename}"
        )


    for i in range(
        data_start,
        len(lines)
    ):

        if lines[i].startswith(
            "#"
        ):

            data_end = i

            break


    if data_end is None:

        data_end = len(
            lines
        )


    return np.array(

        [

            [
                float(v)

                for v in line.split()
            ]

            for line

            in lines[
                data_start:
                data_end
            ]

            if line.strip()

        ],

        dtype=np.float32
    )


# =====================================================================
# 19. DETECTOR GEOMETRY
# =====================================================================

detector_width = float(

    settings[
        "detector_width_m"
    ]
)


detector_height = float(

    settings[
        "detector_height_m"
    ]
)


sample_detector_distance = float(

    settings[
        "sample_detector_distance_m"
    ]
)


original_ncount = int(

    settings[
        "ncount"
    ]
)


print()

print(
    f"External simulations use same ncount = "
    f"{original_ncount:,}"
)


# =====================================================================
# 20. CREATE ANGULAR BIN EDGES FROM MODEL'S ORIGINAL 2THETA GRID
# =====================================================================
#
# Very important:
#
# We do NOT invent a new angular grid.
#
# We force every new diffraction pattern onto EXACTLY the same
# 2theta bins used to train the model.
#
# =====================================================================

theta_step = np.median(

    np.diff(
        theta_reference
    )
)


theta_edges = np.empty(

    len(
        theta_reference
    )
    +
    1,

    dtype=np.float64
)


theta_edges[1:-1] = (

    theta_reference[:-1]
    +
    theta_reference[1:]

) / 2


theta_edges[0] = (

    theta_reference[0]
    -
    theta_step / 2
)


theta_edges[-1] = (

    theta_reference[-1]
    +
    theta_step / 2
)


# =====================================================================
# 21. RADIAL INTEGRATION USING EXACT TRAINING GRID
# =====================================================================

def calculate_radial_profile(
    image
):

    ny, nx = image.shape


    dx = (

        detector_width
        /
        nx
    )


    dy = (

        detector_height
        /
        ny
    )


    x = (

        np.arange(
            nx
        )

        -
        (nx - 1) / 2

    ) * dx


    y = (

        np.arange(
            ny
        )

        -
        (ny - 1) / 2

    ) * dy


    X, Y = np.meshgrid(
        x,
        y
    )


    radius = np.sqrt(

        X**2
        +
        Y**2
    )


    two_theta = np.degrees(

        np.arctan2(

            radius,

            sample_detector_distance
        )
    )


    theta_flat = two_theta.ravel()

    intensity_flat = np.clip(

        image.astype(
            np.float64
        ).ravel(),

        0,

        None
    )


    bin_index = np.digitize(

        theta_flat,

        theta_edges

    ) - 1


    valid = (

        (bin_index >= 0)

        &

        (
            bin_index
            <
            len(
                theta_reference
            )
        )
    )


    bin_index = bin_index[
        valid
    ]


    intensity_flat = intensity_flat[
        valid
    ]


    intensity_sum = np.bincount(

        bin_index,

        weights=intensity_flat,

        minlength=len(
            theta_reference
        )
    )


    pixel_count = np.bincount(

        bin_index,

        minlength=len(
            theta_reference
        )
    )


    intensity_mean = np.divide(

        intensity_sum,

        pixel_count,

        out=np.zeros_like(
            intensity_sum
        ),

        where=(
            pixel_count > 0
        )
    )


    return intensity_mean.astype(
        np.float32
    )


# =====================================================================
# 22. EXACT TRAINING PREPROCESSING
# =====================================================================

def preprocess_profile(
    signal
):

    x = np.asarray(

        signal,

        dtype=np.float64
    ).copy()


    x = np.nan_to_num(

        x,

        nan=0.0,

        posinf=0.0,

        neginf=0.0
    )


    x = np.clip(

        x,

        0,

        None
    )


    positive = x[
        x > 0
    ]


    if positive.size == 0:

        return np.zeros_like(

            x,

            dtype=np.float32
        )


    scale = np.percentile(

        positive,

        INTENSITY_PERCENTILE
    )


    if (

        not np.isfinite(
            scale
        )

        or

        scale <= 0

    ):

        scale = positive.max()


    x = (

        x
        /
        scale
    )


    x = np.clip(

        x,

        0,

        MAX_NORMALIZED_INTENSITY
    )


    x = (

        np.log1p(

            LOG_COMPRESSION
            *
            x

        )

        /

        np.log1p(
            LOG_COMPRESSION
        )
    )


    # exact train-set standardization
    x = (

        x
        -
        train_mean

    ) / train_std


    return x.astype(
        np.float32
    )


# =====================================================================
# 23. INFERENCE HELPER
# =====================================================================

def amp_context():

    if USE_AMP:

        return torch.autocast(

            device_type="cuda",

            dtype=torch.float16
        )

    return contextlib.nullcontext()


def predict_material(
    raw_profile
):

    processed = preprocess_profile(
        raw_profile
    )


    tensor = torch.tensor(

        processed,

        dtype=torch.float32
    )


    tensor = (

        tensor
        .unsqueeze(0)
        .unsqueeze(0)
        .to(
            device
        )
    )


    with torch.no_grad():

        with amp_context():

            logits = model(
                tensor
            )


        probabilities = torch.softmax(

            logits,

            dim=1
        )[0].cpu().numpy()


    predicted_index = int(

        np.argmax(
            probabilities
        )
    )


    return (

        class_names[
            predicted_index
        ],

        probabilities
    )


# =====================================================================
# 24. NUMBER TAG FOR FILENAMES
# =====================================================================

def number_tag(
    value
):

    return (

        f"{value:.5f}"
        .replace(
            ".",
            "p"
        )
    )


# =====================================================================
# 25. RUN NEW EXTERNAL SIMULATIONS
# =====================================================================

results = []

saved_profiles = {}


total_simulations = (

    len(
        class_names
    )

    *
    len(
        unseen_conditions
    )
)


simulation_counter = 0


print()

print("=" * 80)
print("GENERATING COMPLETELY NEW TEST DATA")
print("=" * 80)


for true_material in class_names:


    ncrystal_file = material_to_ncrystal[
        true_material
    ]


    for wavelength, dlambda in unseen_conditions:


        simulation_counter += 1


        sample_name = (

            f"{true_material}"

            f"_lambda_"
            f"{number_tag(wavelength)}"

            f"_dlambda_"
            f"{number_tag(dlambda)}"
        )


        run_dir = (

            runs_dir
            /
            sample_name
        )


        pattern_file = (

            patterns_2d_dir
            /
            f"{sample_name}.npy"
        )


        profile_file = (

            profiles_1d_dir
            /
            f"{sample_name}_profile.npz"
        )


        print()

        print(

            f"["
            f"{simulation_counter:04d}"
            f"/"
            f"{total_simulations:04d}"
            f"] "

            f"{true_material:<15} "

            f"lambda={wavelength:.5f} Å   "

            f"dlambda={dlambda:.5f} Å"
        )


        # =============================================================
        # SIMULATE OR REUSE
        # =============================================================

        if (

            profile_file.exists()

            and

            pattern_file.exists()

            and

            not OVERWRITE_EXISTING

        ):


            image = np.load(
                pattern_file
            )


            saved = np.load(
                profile_file
            )


            radial_profile = saved[
                "intensity_mean"
            ]


            simulation_status = "existing"


        else:


            if run_dir.exists():

                shutil.rmtree(
                    run_dir
                )


            command = [

                mcrun,

                str(
                    instrument_file
                ),

                f"lambda0="
                f"{wavelength:.10f}",

                f"dlambda="
                f"{dlambda:.10f}",

                f"sample_cfg="
                f"{ncrystal_file}",

                "-n",

                str(
                    original_ncount
                ),

                "-d",

                str(
                    run_dir
                )
            ]


            result = subprocess.run(

                command,

                capture_output=True,

                text=True
            )


            if result.returncode != 0:

                print()

                print(
                    "❌ McStas simulation failed"
                )

                print()

                print(
                    result.stdout
                )

                print(
                    result.stderr
                )


                raise RuntimeError(

                    f"\nSimulation failed:\n"

                    f"material = {true_material}\n"

                    f"lambda = {wavelength}\n"

                    f"dlambda = {dlambda}"
                )


            detector_file = (

                run_dir
                /
                "rings.dat"
            )


            if not detector_file.exists():

                raise FileNotFoundError(

                    f"\nDetector file missing:\n"

                    f"{detector_file}"
                )


            image = read_mcstas_2d_intensity(

                detector_file
            )


            radial_profile = calculate_radial_profile(

                image
            )


            if SAVE_2D_PATTERNS:

                np.save(

                    pattern_file,

                    image
                )


            np.savez_compressed(

                profile_file,

                two_theta_deg=
                    theta_reference.astype(
                        np.float32
                    ),

                intensity_mean=
                    radial_profile.astype(
                        np.float32
                    )
            )


            simulation_status = "simulated"


        # =============================================================
        # PREDICT
        # =============================================================

        predicted_material, probabilities = predict_material(

            radial_profile
        )


        true_index = class_to_idx[
            true_material
        ]


        true_probability = float(

            probabilities[
                true_index
            ]
        )


        sorted_indices = np.argsort(

            probabilities
        )[::-1]


        predicted_index = int(

            sorted_indices[
                0
            ]
        )


        second_index = int(

            sorted_indices[
                1
            ]
        )


        confidence = float(

            probabilities[
                predicted_index
            ]
        )


        second_confidence = float(

            probabilities[
                second_index
            ]
        )


        correct = (

            predicted_material
            ==
            true_material
        )


        print(

            f"    TRUE: {true_material:<15} | "

            f"PREDICTED: {predicted_material:<15} | "

            f"confidence={confidence:.4f} | "

            f"{'✓ CORRECT' if correct else '✗ WRONG'}"
        )


        results.append(

            {

                "sample_name":
                    sample_name,

                "true_material":
                    true_material,

                "predicted_material":
                    predicted_material,

                "correct":
                    correct,

                "lambda_A":
                    wavelength,

                "dlambda_A":
                    dlambda,

                "confidence":
                    confidence,

                "true_class_probability":
                    true_probability,

                "second_prediction":
                    class_names[
                        second_index
                    ],

                "second_confidence":
                    second_confidence,

                "status":
                    simulation_status,

                "profile_file":
                    str(
                        profile_file.relative_to(
                            external_test_dir
                        )
                    )
            }
        )


        saved_profiles[
            sample_name
        ] = radial_profile


# =====================================================================
# 26. RESULTS TABLE
# =====================================================================

results_df = pd.DataFrame(
    results
)


results_file = (

    external_test_dir
    /
    "external_test_predictions.csv"
)


results_df.to_csv(

    results_file,

    index=False
)


# =====================================================================
# 27. METRICS
# =====================================================================

y_true_names = results_df[
    "true_material"
].to_numpy()


y_pred_names = results_df[
    "predicted_material"
].to_numpy()


external_accuracy = accuracy_score(

    y_true_names,

    y_pred_names
)


external_balanced_accuracy = balanced_accuracy_score(

    y_true_names,

    y_pred_names
)


external_macro_f1 = f1_score(

    y_true_names,

    y_pred_names,

    average="macro",

    zero_division=0
)


print()

print("=" * 80)
print("EXTERNAL UNSEEN-DATA PERFORMANCE")
print("=" * 80)

print()

print(
    f"Number of completely new patterns : "
    f"{len(results_df)}"
)

print()

print(
    f"Accuracy                         : "
    f"{external_accuracy:.4f}"
)

print(
    f"Balanced accuracy                : "
    f"{external_balanced_accuracy:.4f}"
)

print(
    f"Macro F1                         : "
    f"{external_macro_f1:.4f}"
)

print(
    f"Mean confidence                  : "
    f"{results_df['confidence'].mean():.4f}"
)

print()


# =====================================================================
# 28. PER-MATERIAL PERFORMANCE
# =====================================================================

report = classification_report(

    y_true_names,

    y_pred_names,

    labels=class_names,

    target_names=class_names,

    output_dict=True,

    zero_division=0
)


report_df = pd.DataFrame(
    report
).T


report_df.to_csv(

    external_test_dir
    /
    "external_classification_report.csv"
)


print(
    "Per-material performance:"
)

display(

    report_df.loc[
        class_names
    ]
)


# =====================================================================
# 29. SHOW ALL PREDICTIONS
# =====================================================================

print()

print("=" * 80)
print("INDIVIDUAL PREDICTIONS")
print("=" * 80)

display(

    results_df[
        [

            "true_material",

            "lambda_A",

            "dlambda_A",

            "predicted_material",

            "confidence",

            "second_prediction",

            "second_confidence",

            "correct"

        ]
    ]
)


# =====================================================================
# 30. SHOW ONLY FAILURES
# =====================================================================

wrong_df = results_df[

    ~results_df[
        "correct"
    ]

].copy()


print()

print("=" * 80)
print("MISCLASSIFIED EXAMPLES")
print("=" * 80)


if len(
    wrong_df
) == 0:

    print()

    print(
        "✓ No misclassifications in this external test."
    )

else:

    display(

        wrong_df[
            [

                "true_material",

                "lambda_A",

                "dlambda_A",

                "predicted_material",

                "confidence",

                "second_prediction",

                "second_confidence"

            ]
        ]
    )


# =====================================================================
# 31. CONFUSION MATRIX
# =====================================================================

cm = confusion_matrix(

    y_true_names,

    y_pred_names,

    labels=class_names
)


cm_norm = confusion_matrix(

    y_true_names,

    y_pred_names,

    labels=class_names,

    normalize="true"
)


fig = plt.figure(

    figsize=(
        10,
        8
    )
)


plt.imshow(
    cm_norm
)


plt.colorbar(
    label="Fraction of true class"
)


plt.xticks(

    np.arange(
        len(
            class_names
        )
    ),

    class_names,

    rotation=45,

    ha="right"
)


plt.yticks(

    np.arange(
        len(
            class_names
        )
    ),

    class_names
)


plt.xlabel(
    "Predicted material"
)


plt.ylabel(
    "True material"
)


plt.title(

    "External unseen-condition confusion matrix"
)


for i in range(
    len(
        class_names
    )
):

    for j in range(
        len(
            class_names
        )
    ):

        value = cm_norm[
            i,
            j
        ]


        if value >= 0.01:

            plt.text(

                j,
                i,

                f"{value:.2f}",

                ha="center",

                va="center"
            )


plt.tight_layout()


plt.savefig(

    external_test_dir
    /
    "external_confusion_matrix.png",

    dpi=180
)


plt.show()


# =====================================================================
# 32. ACCURACY BY UNSEEN CONDITION
# =====================================================================

condition_performance = (

    results_df

    .groupby(

        [
            "lambda_A",
            "dlambda_A"
        ]

    )[
        "correct"
    ]

    .mean()

    .reset_index()

    .rename(

        columns={

            "correct":
                "accuracy"
        }
    )
)


print()

print("=" * 80)
print("PERFORMANCE BY NEW LAMBDA / DLAMBDA CONDITION")
print("=" * 80)


display(
    condition_performance
)


# =====================================================================
# 33. ACCURACY BY MATERIAL
# =====================================================================

material_performance = (

    results_df

    .groupby(
        "true_material"
    )[
        "correct"
    ]

    .mean()

    .reset_index()

    .rename(

        columns={

            "correct":
                "accuracy"
        }
    )
)


print()

print("=" * 80)
print("ACCURACY BY MATERIAL")
print("=" * 80)

display(
    material_performance
)


# =====================================================================
# 34. SHOW EXAMPLE UNSEEN DIFFRACTION PATTERNS + PREDICTIONS
# =====================================================================

n_examples = min(

    N_EXAMPLES_TO_PLOT,

    len(
        results_df
    )
)


# Try to spread examples among materials
example_indices = np.linspace(

    0,

    len(
        results_df
    )
    -
    1,

    n_examples,

    dtype=int
)


ncols = 2

nrows = int(

    np.ceil(
        n_examples
        /
        ncols
    )
)


fig, axes = plt.subplots(

    nrows,

    ncols,

    figsize=(
        14,
        3.5
        *
        nrows
    ),

    constrained_layout=True
)


axes = np.asarray(
    axes
).reshape(
    -1
)


for ax, result_index in zip(

    axes,

    example_indices
):


    row = results_df.iloc[
        result_index
    ]


    profile = saved_profiles[
        row[
            "sample_name"
        ]
    ]


    ax.plot(

        theta_reference,

        profile
    )


    correct_symbol = (

        "✓"

        if row[
            "correct"
        ]

        else

        "✗"
    )


    ax.set_title(

        f"{correct_symbol} "

        f"True: {row['true_material']}   |   "
        f"Pred: {row['predicted_material']}\n"

        rf"$\lambda={row['lambda_A']:.4f}$ Å, "

        rf"$\Delta\lambda={row['dlambda_A']:.4f}$ Å   |   "

        f"P={row['confidence']:.3f}"
    )


    ax.set_xlabel(

        r"$2\theta$ [deg]"
    )


    ax.set_ylabel(

        "Mean radial intensity"
    )


    ax.set_yscale(

        "symlog",

        linthresh=1e-12
    )


    ax.grid(
        alpha=0.25
    )


for ax in axes[
    n_examples:
]:

    ax.axis(
        "off"
    )


plt.suptitle(

    "Completely unseen McStas diffraction patterns",

    fontsize=14
)


plt.show()


# =====================================================================
# 35. CONFIDENCE DISTRIBUTION
# =====================================================================

fig = plt.figure(

    figsize=(
        8,
        5
    )
)


plt.hist(

    results_df[
        "confidence"
    ],

    bins=15
)


plt.xlabel(
    "Prediction confidence"
)


plt.ylabel(
    "Number of external samples"
)


plt.title(

    "Confidence on completely unseen simulations"
)


plt.grid(
    alpha=0.25
)


plt.tight_layout()

plt.show()


# =====================================================================
# 36. SAVE SUMMARY
# =====================================================================

summary = {

    "number_of_external_patterns":
        int(
            len(
                results_df
            )
        ),

    "number_of_external_conditions":
        int(
            len(
                unseen_conditions
            )
        ),

    "accuracy":
        float(
            external_accuracy
        ),

    "balanced_accuracy":
        float(
            external_balanced_accuracy
        ),

    "macro_f1":
        float(
            external_macro_f1
        ),

    "mean_confidence":
        float(
            results_df[
                "confidence"
            ].mean()
        ),

    "conditions":
        [

            {

                "lambda_A":
                    float(
                        lam
                    ),

                "dlambda_A":
                    float(
                        dlam
                    )

            }

            for lam, dlam

            in unseen_conditions
        ]
}


with open(

    external_test_dir
    /
    "external_test_summary.json",

    "w"

) as f:

    json.dump(

        summary,

        f,

        indent=4
    )


# =====================================================================
# 37. FINAL
# =====================================================================

print()

print("=" * 80)
print("EXTERNAL TEST COMPLETE")
print("=" * 80)

print()

print(
    "External test directory:"
)

print(
    external_test_dir.resolve()
)

print()

print(
    "Predictions:"
)

print(
    results_file.resolve()
)

print()

print(
    "These lambda/dlambda conditions were NEVER present "
    "in the original training/validation/test dataset."
)

print()

print("=" * 80)